# Jarvis Ear — Kaggle A: Continuous STT

Separate GPU session for Whisper only. **No Groq API key, no voice recording, no OmniVoice in this notebook.**

Enable **GPU T4** and Internet. Add Kaggle Secret `STT_WS_TOKEN` (24+ random characters; enable notebook access). Keep this notebook running while using Jarvis on Windows. Use headphones to prevent the microphone from hearing your speaker.

Run top to bottom. Kaggle sessions are not permanent servers.

## 1. Pull the latest repository

In [ ]:
import os, sys, time, subprocess, site, re
from pathlib import Path
REPO=Path("/kaggle/working/jarvis-realtime-voice")
if (REPO/".git").exists():
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","https://github.com/Godf-Cyb3r-1/jarvis-realtime-voice.git",str(REPO)],check=True)
os.chdir(REPO)
print("Repo:",REPO)

## 2. Install required STT packages if missing (restart session after installation)

In [ ]:
import importlib.util
needed={"faster_whisper":"faster_whisper","fastapi":"fastapi","uvicorn":"uvicorn",
        "numpy":"numpy","httpx":"httpx","dotenv":"dotenv","torch":"torch"}
missing=[name for name,mod in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:",missing)
    subprocess.run([sys.executable,"-m","pip","install","-r",
                    str(REPO/"requirements-realtime-stt.txt")],check=True)
    raise RuntimeError("Restart Python session then rerun all cells")
import torch, ctranslate2
assert torch.cuda.is_available(),"Enable GPU acceleration"
assert ctranslate2.get_cuda_device_count()>=1,"Whisper CUDA unavailable"
print("GPU:",torch.cuda.get_device_name(0))

## 3. Configure microphone recognition and private authentication

In [ ]:
from kaggle_secrets import UserSecretsClient
STT_TOKEN=UserSecretsClient().get_secret("STT_WS_TOKEN")
assert STT_TOKEN and len(STT_TOKEN)>=24,"Add STT_WS_TOKEN in Kaggle Secrets"
os.environ["STT_WS_TOKEN"]=STT_TOKEN
del STT_TOKEN
os.environ.update({
    "STT_MODEL":"medium",  # optional large-v3; higher accuracy costs latency
    "STT_DEVICE":"cuda",
    "STT_GPU_INDEX":"0",
    "STT_COMPUTE":"int8_float16",
    "STT_PARTIAL_INTERVAL":"0.55",
    "ASR_LANGUAGE":"auto",
    "ASR_ALLOWED_LANGUAGES":"en,ta",
    "ASR_BEAM_SIZE":"3",
    "ASR_HOTWORDS":"Jarvis",
    "ASR_MIN_AVG_LOGPROB":"-1.25",
    "SMART_TURN_ENABLED":"0",  # enable only when matching ONNX weights are installed
    "PYTHONUNBUFFERED":"1"
})
libs=[]
for sd in site.getsitepackages():
    for name in ("cublas","cudnn","cuda_runtime"):
        p=Path(sd)/"nvidia"/name/"lib"
        if p.is_dir():libs.append(str(p))
old=os.environ.get("LD_LIBRARY_PATH","")
os.environ["LD_LIBRARY_PATH"]=":".join(libs+([old] if old else []))
print("STT configured. No tokens printed.")

## 4. Syntax checks, start one STT server and verify health

In [ ]:
subprocess.run([sys.executable,"-m","compileall","-q",
    "realtime_stt_server.py","asr_quality.py","realtime_protocol.py"],check=True)
import requests
if "server" in globals() and server and server.poll() is None:
    server.terminate();server.wait(timeout=20)
SERVER_LOG=Path("/kaggle/working/jarvis_stt_server.log")
server_handle=SERVER_LOG.open("w")
server=subprocess.Popen([sys.executable,"-m","uvicorn","realtime_stt_server:app",
    "--host","127.0.0.1","--port","8091"],cwd=REPO,env=os.environ.copy(),
    stdout=server_handle,stderr=subprocess.STDOUT)
health=None
for i in range(180):
    if server.poll() is not None:break
    try:
        x=requests.get("http://127.0.0.1:8091/health",timeout=2)
        if x.ok:health=x.json();break
    except requests.RequestException:pass
    time.sleep(1)
if not health:
    print(SERVER_LOG.read_text(errors="replace")[-4000:])
    raise RuntimeError("STT server failed to start")
print("STT health:",health)

## 5. Create STT tunnel and copy STT_WS_URL to Windows .env

In [ ]:
import re, requests
CLOUDFLARED=Path("/kaggle/working/cloudflared")
if not CLOUDFLARED.is_file():
    subprocess.run(["wget","-q","https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64","-O",str(CLOUDFLARED)],check=True)
CLOUDFLARED.chmod(0o755)
if "tunnel" in globals() and tunnel and tunnel.poll() is None:
    tunnel.terminate()
    tunnel.wait(timeout=15)
TUNNEL_LOG=Path("/kaggle/working/stt_tunnel.log")
tunnel_handle=TUNNEL_LOG.open("w")
tunnel=subprocess.Popen([str(CLOUDFLARED),"tunnel","--url","http://127.0.0.1:8091"],
    stdout=tunnel_handle,stderr=subprocess.STDOUT)
url=None
for _ in range(70):
    if tunnel.poll() is not None:break
    time.sleep(1)
    matches=re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com",
                     TUNNEL_LOG.read_text(errors="replace"))
    if matches:
        url=matches[-1].replace("https://","wss://")+"/stt"
        break
if not url:
    print(TUNNEL_LOG.read_text(errors="replace")[-2000:])
    raise RuntimeError("Tunnel failed. Check Internet and cloudflared logs")
print("STT_WS_URL=",url)
print("Keep the Kaggle secret private; only copy this WSS URL into Windows .env")

## 6. Debug logs (optional)

In [ ]:
print(SERVER_LOG.read_text(errors="replace")[-3000:])
print(TUNNEL_LOG.read_text(errors="replace")[-500:])